In [63]:
import pandas as pd
from datasets import load_dataset
import re
import html
import emoji

In [51]:
ds = load_dataset("Jsevisal/go_emotions_wheel")

In [52]:
df_train = ds['train'].to_pandas()
df_test = ds['test'].to_pandas()
df_val = ds['validation'].to_pandas()

In [71]:
_BRAILLE     = re.compile(r"[\u2800-\u28FF\u2580-\u259F\u2580-\u259F]+")
_UNICODE_ART = re.compile(r"[\u2500-\u257F\u2600-\u26FF\u2700-\u27BF]+")
_URL         = re.compile(r"https?://\S+|www\.\S+")
_MENTION     = re.compile(r"@\w+")
_WHITESPACE  = re.compile(r"\s+")
_MD_TABLE_ROW = re.compile(r"(\|[^\n]+\||\|[-:\s|]+\|)", re.MULTILINE)
_STRAY_PIPE   = re.compile(r"\s*\|\s*")
_DUP_EMOJI    = re.compile(r"(:[a-z_]+:)(\s*\1)+")
_REP_PUNCT    = re.compile(r"([^\w\s])\1{3,}")

In [72]:
def clean_for_neural(text: str) -> str:
    # 1. Decode HTML entities
    text = html.unescape(text)
 
    # 2. Remove braille / unicode block-drawing art
    text = _BRAILLE.sub(" ", text)
    text = _UNICODE_ART.sub(" ", text)
 
    # 3. Remove markdown tables
    text = _MD_TABLE_ROW.sub(" ", text)
    text = _STRAY_PIPE.sub(" ", text)
 
    # 4. Strip URLs and @mentions
    text = _URL.sub(" ", text)
    text = _MENTION.sub(" ", text)
 
    # 5. Collapse repeated punctuation  (++++++++++++ → +, !!!!! → !)
    text = _REP_PUNCT.sub(r"\1", text)
 
    # 6. Convert emojis to :text: form  (🔥 → :fire:, 👏 → :clapping_hands:)
    text = emoji.demojize(text, delimiters=(":", ":"))
 
    # 7. Deduplicate consecutive identical emoji tokens
    #    e.g. ":fire: :fire: :fire:" → ":fire:"
    text = _DUP_EMOJI.sub(r"\1", text)
 
    # 8. Normalize whitespace
    text = _WHITESPACE.sub(" ", text).strip()
 
    return text

In [73]:
def clean_df(df) -> pd.DataFrame:
    df["text_neural"] = df["text"].apply(clean_for_neural)
     
    # Flag rows that became empty or too short after cleaning
    df["_empty_after_clean"] = df["text_neural"].str.strip().eq("")
    df["_token_count"]       = df["text_neural"].str.split().str.len()
     
    empty_count = df["_empty_after_clean"].sum()
    short_count = (df["_token_count"] < 3).sum()
     
    print(f"Rows emptied by cleaning : {empty_count}")
    print(f"Rows with < 3 tokens     : {short_count}")
     
    # Optional: drop empty rows (uncomment if you want to remove them)
    # df = df[~df["_empty_after_clean"]].reset_index(drop=True)
     
    # Drop helper columns before saving / passing downstream
    df_clean = df.drop(columns=["_empty_after_clean", "_token_count", "text"])
    df_clean = df_clean.rename(columns = {"text_neural": "text"})
    return df_clean

In [77]:
df_train = clean_df(df_train)
df_test = clean_df(df_test)
df_val = clean_df(df_val)

Rows emptied by cleaning : 0
Rows with < 3 tokens     : 1055
Rows emptied by cleaning : 0
Rows with < 3 tokens     : 119
Rows emptied by cleaning : 0
Rows with < 3 tokens     : 131


In [82]:
total = df_train.shape[0] + df_test.shape[0] + df_val.shape[0]

In [85]:
print(df_train.shape[0] * 100 / total,
df_test.shape[0] * 100 / total,
df_val.shape[0] * 100 / total)

79.99926284945543 10.001290013452998 9.999447137091572


In [86]:
sub_val = df_val.sample(frac = 0.05, random_state = 42)

In [87]:
df_val = df_val.drop(sub_val.index)

In [88]:
df_test = pd.concat([df_test, sub_val], ignore_index = True)

In [89]:
print(df_train.shape[0] * 100 / total,
df_test.shape[0] * 100 / total,
df_val.shape[0] * 100 / total)

79.99926284945543 10.500709507399149 9.50002764314542


In [75]:
df_train.to_parquet("df_train.parquet", index=False)
df_test.to_parquet("df_test.parquet", index=False)
df_val.to_parquet("df_val.parquet", index=False)

In [76]:
df_train.head()

,labels,id,text
0,[5],eebbqej,My favourite food is anything I didn't have to...
1,[5],ed00q6i,"Now if he does off himself, everyone will thin..."
2,[7],eezlygj,WHY THE FUCK IS BAYLESS ISOING
3,[2],ed7ypvh,To make her feel threatened
4,[7],ed0bdzj,Dirty Southern Wankers
